# Training and Test Evaluation

This notebook shows SAC validation history and compares Random, GeneticBatch, pretrained discrete SAC, and the adaptive online hybrid on the eight test datasets.

In [ ]:
from collections import defaultdict
from pathlib import Path
import csv
import os

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-vec-cache')

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'evaluation.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root.')


PROJECT_ROOT = find_project_root()
RESULTS_ROOT = PROJECT_ROOT / 'outputs' / 'evaluation'
HISTORY_FILE = PROJECT_ROOT / 'outputs' / 'models' / 'discrete_sac' / 'validation_history.csv'
HYBRID_FILE = PROJECT_ROOT / 'outputs' / 'online' / 'test_hybrid_results.csv'
SUMMARY_FILES = {
    'Random': RESULTS_ROOT / 'random' / 'test_random_summary.csv',
    'GeneticBatch': RESULTS_ROOT / 'genetic' / 'test_genetic_summary.csv',
    'SAC_Pretrained': RESULTS_ROOT / 'sac_pretrained' / 'test_sac_pretrained_summary.csv',
}

print('Project:', PROJECT_ROOT)

## SAC Training History

In [ ]:
if not HISTORY_FILE.exists():
    raise FileNotFoundError(f'Missing training history: {HISTORY_FILE}')

with HISTORY_FILE.open(newline='', encoding='utf-8') as handle:
    history = list(csv.DictReader(handle))

checks = np.arange(1, len(history) + 1)
validation_reward = np.array([float(row['validation_average_reward']) for row in history])
miss_rate = np.array([float(row['validation_deadline_miss_rate']) for row in history])
loss_rate = np.array([float(row['validation_packet_loss_rate']) for row in history])
best_index = int(np.argmax(validation_reward))
best = history[best_index]

print(
    f"Best checkpoint selection: epoch={best['epoch']} after={best['dataset']} "
    f"reward={float(best['validation_average_reward']):.4f} "
    f"miss={float(best['validation_deadline_miss_rate']):.4f} "
    f"loss={float(best['validation_packet_loss_rate']):.4f}"
)

figure, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(checks, validation_reward, color='#287271')
axes[0].scatter(checks[best_index], validation_reward[best_index], color='#C44536', label='saved best')
axes[0].set_title('Validation reward during pretraining')
axes[0].set_ylabel('mean bounded reward')
axes[0].legend()
axes[1].plot(checks, miss_rate, color='#6A4C93', label='deadline-miss rate')
axes[1].plot(checks, loss_rate, color='#C44536', label='packet-loss rate')
axes[1].set_title('Validation failure rates')
axes[1].set_ylabel('rate')
axes[1].legend()
for axis in axes:
    axis.set_xlabel('validation check')
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Test Summary

The online hybrid is an adaptive prequential result, so it is shown alongside the frozen algorithms but should be labeled as adaptive in the report.

In [ ]:
def load_csv(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'Missing result file: {path}')
    with path.open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


summaries = {name: load_csv(path) for name, path in SUMMARY_FILES.items()}

hybrid_groups = defaultdict(list)
for row in load_csv(HYBRID_FILE):
    hybrid_groups[row['scenario_group']].append(row)

hybrid_summary = []
for scenario, rows in hybrid_groups.items():
    count = len(rows)
    hybrid_summary.append({
        'scenario_group': scenario,
        'total_tasks': count,
        'deadline_misses': sum(row['deadline_missed'].lower() == 'true' for row in rows),
        'packet_losses': sum(row['packet_lost'].lower() == 'true' for row in rows),
        'avg_latency': sum(float(row['latency']) for row in rows) / count,
        'avg_energy': sum(float(row['total_system_energy']) for row in rows) / count,
    })
summaries['OnlineHybrid'] = hybrid_summary

for algorithm, rows in summaries.items():
    print(f'\n{algorithm}')
    for row in rows:
        print(
            f"{row['scenario_group']:12s} tasks={int(row['total_tasks']):6d} "
            f"miss={int(row['deadline_misses']):5d} loss={int(row['packet_losses']):4d} "
            f"latency={float(row['avg_latency']):8.4f} energy={float(row['avg_energy']):7.4f}"
        )

## Required Metrics

In [ ]:
SCENARIOS = ['BASE', 'RAIN', 'SNOW', 'FOG', 'FAST_MIXED', 'SLOW_MIXED', 'RANDOM_MIX_1', 'RANDOM_MIX_2']
ALGORITHMS = ['Random', 'GeneticBatch', 'SAC_Pretrained', 'OnlineHybrid']
COLORS = ['#4F83CC', '#62A87C', '#B86B4B', '#7A5AA6']
summary_by_algorithm = {
    algorithm: {row['scenario_group']: row for row in rows}
    for algorithm, rows in summaries.items()
}


def plot_metric(metric: str, title: str, ylabel: str):
    x = np.arange(len(SCENARIOS))
    width = 0.2
    figure, axis = plt.subplots(figsize=(15, 4.8))
    center = (len(ALGORITHMS) - 1) / 2
    for index, (algorithm, color) in enumerate(zip(ALGORITHMS, COLORS)):
        values = [float(summary_by_algorithm[algorithm][scenario][metric]) for scenario in SCENARIOS]
        axis.bar(x + (index - center) * width, values, width, label=algorithm, color=color)
    axis.set_title(title)
    axis.set_ylabel(ylabel)
    axis.set_xticks(x)
    axis.set_xticklabels(SCENARIOS, rotation=20, ha='right')
    axis.legend()
    axis.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_metric('deadline_misses', 'Deadline Misses', 'count')

In [ ]:
plot_metric('packet_losses', 'Packet Losses', 'count')

In [ ]:
plot_metric('avg_latency', 'Average Latency', 'seconds')

In [ ]:
plot_metric('avg_energy', 'Average Total System Energy', 'joules')